# Direct labels

**Label the lines, not the legend.**

A legend makes the reader do a matching exercise: find the colour, carry it across the chart, find the line. Putting the name next to the line removes that work entirely.

**What it shows:**

- a legend replaced by labels at the end of each line
- ax.annotate with data coordinates, and how to stop labels overlapping
- why this matters more as the number of series grows

---

*Chapter:* `annotation` — labels, highlighting, and titles that say something  
*Run the cells in order.* Every figure is also written to `viz/output/annotation/`, which is what the Streamlit gallery (`viz/project/gallery.py`) reads.


## Setup

These lines are how every notebook in the folder finds `vizkit.py`, which holds the save helpers and the seeded sample data. The data is seeded on purpose: your figures should come out identical to everyone else's.

`save()` writes each figure into `viz/output/` **and** leaves it on screen here. The trailing `;` on those calls only stops the notebook echoing the path it returns.


In [6]:
%matplotlib inline

# A notebook has no __file__, so find viz/ by walking up from this
# notebook's own folder until vizkit.py turns up.
import sys
from pathlib import Path

VIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
           if (p / "vizkit.py").exists())
sys.path.insert(0, str(VIZ))

import matplotlib.pyplot as plt

from vizkit import save, sales

# Where save() files this lesson's output: viz/output/annotation/
LESSON = "annotation/direct_labels"


## The data

Four regions, plus one fixed colour each so the same region keeps the same colour in every figure.


In [7]:
wide = sales().pivot(index="month", columns="region", values="sales")
COLOURS = {"North": "#0072B2", "South": "#E69F00",
           "East": "#009E73", "West": "#CC79A7"}


## 1. Legend versus direct labels

Time yourself answering 'how did East end up?' on each panel. The legend version makes you carry a colour across the chart and back; the labelled version already answered it.


In [8]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4.2))

for region in wide.columns:
    left.plot(wide.index, wide[region], color=COLOURS[region], label=region, lw=2)
left.legend(title="region")
left.set_title("Legend: find the colour, then find the line")
left.set_ylabel("sales")

for region in wide.columns:
    right.plot(wide.index, wide[region], color=COLOURS[region], lw=2)
    # Put the name at the line's last point, nudged right and centred.
    right.annotate(
        region,
        xy=(wide.index[-1], wide[region].iloc[-1]),
        xytext=(6, 0), textcoords="offset points",
        color=COLOURS[region], fontsize=10, fontweight="bold",
        va="center",
    )
# Leave room on the right for the labels, or they fall off the edge.
right.set_xlim(wide.index[0], wide.index[-1] + (wide.index[-1] - wide.index[-4]))
right.set_title("Direct labels: the name IS the legend")
right.set_ylabel("sales")

fig.tight_layout()
save(fig, LESSON, "legend-vs-direct");


  saved  viz/output/annotation/direct_labels-legend-vs-direct.png


## 2. When the ends are too close together

Direct labels fail when two lines end close together. This is the cheap fix: sort by final value and push each label up until it clears the one below. Since you are writing the label anyway, put the final number in it too.


In [9]:
# Nudge overlapping labels apart, cheaply: sort by value and enforce a gap.
final = wide.iloc[-1].sort_values()
gap = (wide.to_numpy().max() - wide.to_numpy().min()) * 0.07

positions = {}
last = -1e9
for region, value in final.items():
    position = max(value, last + gap)
    positions[region] = position
    last = position

fig, ax = plt.subplots(figsize=(8, 4.2))
for region in wide.columns:
    ax.plot(wide.index, wide[region], color=COLOURS[region], lw=2)
    ax.annotate(f"{region}  {wide[region].iloc[-1]:.0f}",
                xy=(wide.index[-1], positions[region]),
                xytext=(8, 0), textcoords="offset points",
                color=COLOURS[region], fontsize=9, fontweight="bold", va="center")

ax.set_xlim(wide.index[0], wide.index[-1] + (wide.index[-1] - wide.index[-5]))
ax.set_title("Labels nudged apart, and carrying the final value too")
ax.set_ylabel("sales")
fig.tight_layout()
save(fig, LESSON, "nudged-labels");


  saved  viz/output/annotation/direct_labels-nudged-labels.png


## Rules of thumb

```text
A legend is a lookup table. A label is an answer.
Put the name where the line ends, and delete the legend.
```


## Try it yourself

Edit the cells above and re-run them — that is what the notebook is for.

1. Delete the `set_xlim` line in section 1 and re-run. Where do the labels go, and why does `bbox_inches='tight'` not fully rescue it?
2. Change `gap` in section 2 to 0.01 and to 0.3. What is the smallest gap that still separates all four labels?
3. Label the lines at their *start* instead of their end. Which reads better for this data, and would that change with 20 series?


In [5]:
# your turn


---

**Previous:** [`color/rainbow`](../color/rainbow.ipynb)  
**Next:** [`annotation/highlight`](highlight.ipynb)
